# IMPORTS

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract_all, regexp_replace,col, lit, size,  explode,split, element_at,col,udf,length,make_date,expr,weekofyear,collect_list,explode,max,min,lag,log,avg,when,abs,count,std,stddev,sum
from pyspark.sql.types import StringType
import json
from pyspark.sql import Window
import pandas as pd
import datetime

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

# Data read

In [0]:
feature_store_technical  = spark.read.format('delta').load("/Volumes/market-mood/features/feature_store/") 
spotify = spark.read.format('delta').load('/Volumes/market-mood/processed/spotify_aggregated')

# Join

In [0]:
spotify = spotify.groupBy('date').agg(
    (sum(col("valence") * col("weight")) / sum(col("weight"))).alias("weighted_valence"),
    (sum(col("energy") * col("weight")) / sum(col("weight"))).alias("weighted_energy"),
    (sum(col("danceability") * col("weight")) / sum(col("weight"))).alias("weighted_danceability"),
    (sum(col("acousticness") * col("weight")) / sum(col("weight"))).alias("weighted_acousticness"),
    (sum(col("mode") * col("weight")) / sum(col("weight"))).alias("weighted_mode"),
    (sum(col("instrumentalness") * col("weight")) / sum(col("weight"))).alias("weighted_instrumentalness"),
    (sum(col("key") * col("weight")) / sum(col("weight"))).alias("weighted_key"),
    (sum(col("tempo") * col("weight")) / sum(col("weight"))).alias("weighted_tempo")
)


In [0]:
feature_store_spotify = feature_store_technical.join(
    spotify.select("date", "weighted_valence", "weighted_energy", 
                   "weighted_danceability", "weighted_mode","weighted_instrumentalness","weighted_key","weighted_acousticness", "weighted_tempo"),
    on=["date"],
    how="inner"
)

feature_store_spotify.printSchema()
feature_store_spotify.count()

# Load

In [0]:
(feature_store_spotify
    .write
    .option('overwriteSchema', 'true')
    .format("delta")
    .mode("overwrite")
    .partitionBy("ticker", "year")
    .save("/Volumes/market-mood/features/spotify_store"))